# Machine Learning Pipeline

In this notebook, we will train Machine Learning models to predict **Accident Severity** using the Indian Road Accident dataset.

**Pipeline Steps:**
1. Data Loading and Cleaning
2. Categorical Feature Encoding
3. Feature Scaling & Train-Test Split
4. Model Training (Random Forest & XGBoost)
5. Model Evaluation
6. Saving the Best Model

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

### 1. Data Loading and Cleaning
Let's load the data, strip any whitespace from column names, and drop columns that are too specific (like City Name, Location Details) or that leak future information (like Casualties).

In [2]:
# Load dataset
df = pd.read_csv('../data/accident_prediction_india.csv')
df.columns = df.columns.str.strip()

# Check for missing values
print("Missing before clean:\n", df.isnull().sum())

# Fill missing values with Mode (since most features are categorical)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Drop fields we won't use for generic risk prediction 
# (e.g., Casualties is an *outcome*, not a feature. Latitude/Longitude might be too complex for a pure tabular tree right now unless binned).

drop_cols = ['Casualties', 'State Name', 'City Name', 'Accident Location Details', 'Police Station Context', 'Year']
remaining_cols = [c for c in df.columns if c not in drop_cols]
df = df[remaining_cols]

print("\nFeatures to use for modeling:", df.columns.tolist())

Missing before clean:
 State Name                       0
City Name                        0
Year                             0
Month                            0
Day of Week                      0
Time of Day                      0
Accident Severity                0
Number of Vehicles Involved      0
Vehicle Type Involved            0
Number of Casualties             0
Number of Fatalities             0
Weather Conditions               0
Road Type                        0
Road Condition                   0
Lighting Conditions              0
Traffic Control Presence       716
Speed Limit (km/h)               0
Driver Age                       0
Driver Gender                    0
Driver License Status          975
Alcohol Involvement              0
Accident Location Details        0
dtype: int64

Features to use for modeling: ['Month', 'Day of Week', 'Time of Day', 'Accident Severity', 'Number of Vehicles Involved', 'Vehicle Type Involved', 'Number of Casualties', 'Number of Fatalities'

### 2. Categorical Feature Encoding
Most of our data is categorical strings. `RandomForest` and `XGBoost` need numeric inputs.
We will use LabelEncoding for our target (`Accident Severity`) and One-Hot Encoding or Label Encoding for features.

In [3]:
# Target Variable Encoding
target = 'Accident Severity'
le_target = LabelEncoder()
df[target] = le_target.fit_transform(df[target])

# Let's print the mapping:
severity_mapping = dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))
print("Target Mapping:", severity_mapping)

# Feature Encoding (Label Encoding for simplicity across all categorical columns)
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.head()

Target Mapping: {'Fatal': 0, 'Minor': 1, 'Serious': 2}


,Month,Day of Week,Time of Day,Accident Severity,Number of Vehicles Involved,Vehicle Type Involved,Number of Casualties,Number of Fatalities,Weather Conditions,Road Type,Road Condition,Lighting Conditions,Traffic Control Presence,Speed Limit (km/h),Driver Age,Driver Gender,Driver License Status,Alcohol Involvement
0,8,1,604,2,5,3,0,4,2,0,3,0,2,61,66,1,1,1
1,4,6,693,1,5,5,5,4,2,2,1,3,2,92,60,1,1,1
2,8,6,1020,1,5,4,6,5,1,0,2,1,2,120,26,0,1,0
3,6,2,23,1,3,1,10,5,3,1,1,0,1,76,34,0,1,1
4,1,4,113,1,5,3,7,1,1,2,3,3,2,115,30,1,1,0


### 3. Feature Scaling & Train-Test Split

In [4]:
X = df.drop(target, axis=1)
y = df[target]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

Training Set Shape: (2400, 17)
Testing Set Shape: (600, 17)


### 4. Model Training

In [5]:
# 1. Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# 2. XGBoost
print("Training XGBoost...")
xgb_model = XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

Training Random Forest...


Training XGBoost...


### 5. Model Evaluation

In [6]:
print("--- Random Forest Results ---")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds, target_names=le_target.classes_))

print("\n--- XGBoost Results ---")
print(f"Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
print(classification_report(y_test, xgb_preds, target_names=le_target.classes_))

--- Random Forest Results ---
Accuracy: 0.3367
              precision    recall  f1-score   support

       Fatal       0.31      0.32      0.32       197
       Minor       0.36      0.34      0.35       207
     Serious       0.34      0.34      0.34       196

    accuracy                           0.34       600
   macro avg       0.34      0.34      0.34       600
weighted avg       0.34      0.34      0.34       600


--- XGBoost Results ---
Accuracy: 0.3200
              precision    recall  f1-score   support

       Fatal       0.32      0.32      0.32       197
       Minor       0.33      0.31      0.32       207
     Serious       0.31      0.33      0.32       196

    accuracy                           0.32       600
   macro avg       0.32      0.32      0.32       600
weighted avg       0.32      0.32      0.32       600



### 6. Saving the Best Model & Components
We will save XGBoost as it typically performs better. We must also save the `scaler` and `label_encoders` so the Flask Backend API knows how to process raw user inputs!

In [7]:
import joblib

# Save Model
joblib.dump(xgb_model, '../models/risk_model.pkl')

# Save Scaler
joblib.dump(scaler, '../models/scaler.pkl')

# Save the Label Encoders (for the frontend/API alignment)
joblib.dump(label_encoders, '../models/label_encoders.pkl')

print("Model and preprocessors successfully saved to the /models/ folder!")

Model and preprocessors successfully saved to the /models/ folder!
